# 02 Generated first-person view

Turns stage 3's Gaussian first-person render into footage with SDXL inpainting (geometry-guided), and compares it with LTX-Video generating the player's view from the broadcast frame alone (the video-generation baseline). Runs stages 1 and 3 first if their outputs aren't on this machine. Needs a GPU runtime (A100 or L4). Downloads about 20 GB of model weights the first time.

In [ ]:
![ -d throughball ] || git clone -q -b claude/project-thread-0z09g6 https://github.com/ttkgas/throughball.git
%cd /content/throughball
!git pull -q

In [ ]:
# Stages 1 and 3, skipped if already done on this machine
!if [ ! -f outputs/stage1/021/cameras.json ]; then bash setup.sh && third_party/sn-gamestate/.venv/bin/python scripts/get_clip.py SNGS-021 && MPLBACKEND=Agg third_party/sn-gamestate/.venv/bin/tracklab --config-dir configs -cn throughball_setup && third_party/sn-gamestate/.venv/bin/python scripts/export_stage1.py; fi
!if ! ls outputs/stage3/021/pov_*/cameras.json >/dev/null 2>&1; then pip install -q gsplat ultralytics imageio-ffmpeg ninja && python -u scripts/gaussians_pov.py --vid 021; fi

In [ ]:
!pip install -q -U diffusers accelerate sentencepiece imageio-ffmpeg 'transformers<5'

In [ ]:
!python -u scripts/generate_pov.py --vid 021

In [ ]:
import json
from IPython.display import Image, Video, display
print(json.dumps(json.load(open('outputs/stage4/021/metrics.json')), indent=1))
display(Image('outputs/stage4/021/preview.png'))
Video('outputs/stage4/021/compare.mp4', embed=True, width=960)